In [1]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

In [2]:
import os
from dotenv import load_dotenv
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

load_dotenv()


# =========================
# 1. 평가 데이터 (도메인 반영)
# =========================

eval_data = {
    "question": [
        "배달 소비가 너무 많은데 줄이는 방법 알려줘",
        "정부에서 지원해주는 절약 정책 뭐 있어?",
        "카페 소비 줄이는 현실적인 방법 알려줘",
        "야간 소비가 많은데 어떻게 고치지?",
    ],
    "answer": [
        "최근 배달 소비가 반복되고 있습니다. 주 2회 이하로 제한하고 밀키트나 냉동식품을 대체 수단으로 활용하는 것이 좋습니다.",
        "청년 대상 자산형성 지원 정책으로 청년도약계좌, 주거지원 정책 등이 있습니다.",
        "카페 소비를 줄이기 위해 텀블러 사용, 편의점 커피 대체, 주간 예산 설정이 효과적입니다.",
        "야간 소비는 충동 소비일 가능성이 높습니다. 결제 전 10분 대기 규칙을 적용하는 것이 좋습니다.",
    ],
    "contexts": [
        [
            "배달 소비는 반복 소비 패턴으로 이어지기 쉽다.",
            "외식비 절약을 위해 밀키트, 냉동식품 활용이 권장된다.",
        ],
        [
            "청년 자산형성 지원 정책으로 청년도약계좌가 있다.",
            "주거지원 및 청년 대상 금융지원 정책이 제공된다.",
        ],
        [
            "카페 소비는 소액 반복 지출의 대표 사례이다.",
            "텀블러 사용 및 대체 소비는 절약에 효과적이다.",
        ],
        [
            "야간 소비는 충동 소비 및 스트레스 소비와 관련이 있다.",
            "충동 소비를 줄이기 위해 결제 전 대기 전략이 효과적이다.",
        ],
    ],
    "ground_truth": [
        "배달 소비는 횟수 제한과 대체 소비 전략이 필요하다.",
        "청년 대상 금융 지원 정책을 활용할 수 있다.",
        "카페 소비는 대체 소비와 예산 설정으로 줄일 수 있다.",
        "야간 소비는 충동 소비 억제 전략이 필요하다.",
    ],
}

dataset = Dataset.from_dict(eval_data)


# =========================
# 2. RAG 평가 실행
# =========================

result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
)

df = result.to_pandas()

print(df)

df.to_csv("rag_finance_eval.csv", index=False, encoding="utf-8-sig")

print("✅ RAG 평가 완료 (rag_finance_eval.csv 저장)")

c:\Users\user\catcher\catcher-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_11584\1751350555.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_11584\1751350555.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_11584\1751350555.py:6: DeprecationWarning: Importing context_pre

                 user_input  \
0  배달 소비가 너무 많은데 줄이는 방법 알려줘   
1    정부에서 지원해주는 절약 정책 뭐 있어?   
2     카페 소비 줄이는 현실적인 방법 알려줘   
3       야간 소비가 많은데 어떻게 고치지?   

                                  retrieved_contexts  \
0  [배달 소비는 반복 소비 패턴으로 이어지기 쉽다., 외식비 절약을 위해 밀키트, 냉...   
1  [청년 자산형성 지원 정책으로 청년도약계좌가 있다., 주거지원 및 청년 대상 금융지...   
2  [카페 소비는 소액 반복 지출의 대표 사례이다., 텀블러 사용 및 대체 소비는 절약...   
3  [야간 소비는 충동 소비 및 스트레스 소비와 관련이 있다., 충동 소비를 줄이기 위...   

                                            response  \
0  최근 배달 소비가 반복되고 있습니다. 주 2회 이하로 제한하고 밀키트나 냉동식품을 ...   
1        청년 대상 자산형성 지원 정책으로 청년도약계좌, 주거지원 정책 등이 있습니다.   
2  카페 소비를 줄이기 위해 텀블러 사용, 편의점 커피 대체, 주간 예산 설정이 효과적...   
3  야간 소비는 충동 소비일 가능성이 높습니다. 결제 전 10분 대기 규칙을 적용하는 ...   

                        reference  faithfulness  answer_relevancy  \
0   배달 소비는 횟수 제한과 대체 소비 전략이 필요하다.      0.666667               NaN   
1       청년 대상 금융 지원 정책을 활용할 수 있다.      1.000000               NaN   
2  카페 소비는 대체 소비와 예산 설정으로 줄일 수 있다.      0.500000               NaN   
3     